In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np
import gc

# --------------------
# Dataset Paths Configuration
# --------------------

train_csv_path = "train.csv"
train_image_folder = "train"

val_csv_path = "val.csv"
val_image_folder = "val"

test_csv_path = "test.csv"
test_image_folder = "test"

# --------------------
# Dataset Preparation Function
# --------------------

def prepare_dataset(csv_path, image_folder, dataset_name="Dataset"):
    """
    Prepare dataset by reading CSV and validating image paths
    
    Args:
        csv_path: Path to CSV file
        image_folder: Path to folder containing images
        dataset_name: Name for logging purposes
    
    Returns:
        DataFrame with valid image filepaths
    """
    df = pd.read_csv(csv_path)
    extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    
    valid_rows = []
    for _, row in df.iterrows():
        image_id = row['image_id']
        for ext in extensions:
            potential_path = os.path.join(image_folder, f"{image_id}{ext}")
            if os.path.exists(potential_path):
                row['filepath'] = potential_path
                valid_rows.append(row)
                break
    
    filtered_df = pd.DataFrame(valid_rows)
    print(f"{dataset_name}: {len(filtered_df)} valid images found (out of {len(df)} total)")
    
    return filtered_df

print("Preparing datasets...")
print("-" * 50)
train_df = prepare_dataset(train_csv_path, train_image_folder, "Training Set")
val_df = prepare_dataset(val_csv_path, val_image_folder, "Validation Set")
test_df = prepare_dataset(test_csv_path, test_image_folder, "Test Set")

print("-" * 50)
print(f"Dataset Summary:")
print(f"  Train: {len(train_df)} samples")
print(f"  Validation: {len(val_df)} samples")
print(f"  Test: {len(test_df)} samples")
print(f"  Total: {len(train_df) + len(val_df) + len(test_df)} samples")
print("-" * 50)

fiber_columns = ['cotton', 'wool', 'polyester', 'linen', 'silk', 'other_fibres']

# --------------------------
# Dataset Class
# --------------------------

class FabricBLIPDataset(Dataset):
    def __init__(self, df):
        self.data = df.reset_index(drop=True)
        self.fibers = fiber_columns

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        try:
            image = Image.open(row['filepath']).convert("RGB")
            # Normalize fiber values to 0-1 range and handle NaN
            target = []
            for f in self.fibers:
                val = row[f]
                if pd.isna(val):
                    target.append(0.0)
                else:
                    # Always divide by 100 to convert percentage to 0-1 range
                    normalized_val = float(val) / 100.0
                    target.append(max(0.0, min(1.0, normalized_val)))  # Clamp to [0,1]

            target = torch.tensor(target, dtype=torch.float32)
            return {"image": image, "target": target}
        except Exception as e:
            print(f"Error loading image {row['filepath']}: {e}")
            # Return dummy data in case of error
            dummy_image = Image.new('RGB', (224, 224), color='white')
            dummy_target = torch.zeros(len(self.fibers), dtype=torch.float32)
            return {"image": dummy_image, "target": dummy_target}

# --------------------------
# Collate Function
# --------------------------

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

def collate_fn(batch):
    images = [item["image"] for item in batch]
    targets = torch.stack([item["target"] for item in batch])
    prompts = ["What is the fabric composition?"] * len(images)

    try:
        inputs = processor(images=images, text=prompts, return_tensors="pt", padding=True)
        return inputs, targets
    except Exception as e:
        print(f"Error in collate_fn: {e}")
        dummy_inputs = processor(images=[Image.new('RGB', (224, 224), color='white')],
                               text=["What is the fabric composition?"],
                               return_tensors="pt", padding=True)
        dummy_targets = torch.zeros((1, len(fiber_columns)), dtype=torch.float32)
        return dummy_inputs, dummy_targets

# --------------------------
# Dataloaders with drop_last=False to ensure all samples are processed
# --------------------------

train_dataset = FabricBLIPDataset(train_df)
val_dataset = FabricBLIPDataset(val_df)
test_dataset = FabricBLIPDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn, drop_last=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn, drop_last=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn, drop_last=False, num_workers=0)

print(f"DataLoader Info:")
print(f"  Train batches: {len(train_loader)} (covers {len(train_dataset)} samples)")
print(f"  Val batches: {len(val_loader)} (covers {len(val_dataset)} samples)")
print(f"  Test batches: {len(test_loader)} (covers {len(test_dataset)} samples)")
print("-" * 50)

# --------------------------
# Model with LoRA and Regression Head
# --------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

blip = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float32
).to(device)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "out_proj",
        "fc1", "fc2",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
)

print("Applying LoRA to BLIP2 model...")
blip = get_peft_model(blip, lora_config)

total_params = sum(p.numel() for p in blip.parameters())
trainable_params = sum(p.numel() for p in blip.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable parameters ratio: {trainable_params/total_params:.4f}")

class Blip2WithLoRARegressor(nn.Module):
    def __init__(self, blip_model):
        super().__init__()
        self.blip = blip_model
        hidden_size = blip_model.base_model.language_model.config.hidden_size

        self.regressor = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 6),
            nn.Sigmoid()
        ).to(torch.float32)

    def forward(self, pixel_values, input_ids, attention_mask):
        pixel_values = pixel_values.float()

        vision_outputs = self.blip.base_model.vision_model(pixel_values=pixel_values)
        image_embeds = vision_outputs[0]

        image_attention_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)
        query_tokens = self.blip.base_model.query_tokens.expand(image_embeds.shape[0], -1, -1)

        query_outputs = self.blip.base_model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True,
        )
        query_output = query_outputs.last_hidden_state

        language_model_inputs = self.blip.base_model.language_projection(query_output)

        pooled_output = language_model_inputs[:, 0, :]

        return self.regressor(pooled_output)

model = Blip2WithLoRARegressor(blip).to(device)
model = model.float()

print("Model created with LoRA and regression head in float32 precision")

# --------------------------
# Evaluation Functions
# --------------------------

def calculate_accuracy_metrics(predictions, targets, tolerance=0.1):
    """
    Calculate various accuracy metrics for regression task
    """
    predictions = predictions.cpu().numpy() * 100
    targets = targets.cpu().numpy() * 100

    mae = mean_absolute_error(targets.flatten(), predictions.flatten())

    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)

    r2 = r2_score(targets.flatten(), predictions.flatten())

    abs_diff = np.abs(predictions - targets)
    tolerance_accuracy = np.mean(abs_diff <= (tolerance * 100)) * 100

    fiber_metrics = {}
    for i, fiber in enumerate(fiber_columns):
        fiber_mae = mean_absolute_error(targets[:, i], predictions[:, i])
        fiber_r2 = r2_score(targets[:, i], predictions[:, i])
        fiber_metrics[fiber] = {'MAE': fiber_mae, 'R²': fiber_r2}

    return {
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2,
        'Tolerance_Accuracy': tolerance_accuracy,
        'Fiber_Metrics': fiber_metrics
    }

def evaluate_model(model, dataloader, loss_fn, device, dataset_name="Test"):
    """Evaluate model and return detailed metrics"""
    model.eval()
    all_predictions = []
    all_targets = []
    losses = []
    processed_samples = 0

    print(f"\nEvaluating on {dataset_name} set...")

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            try:
                inputs = {k: v.to(device).float() if v.dtype == torch.float16 else v.to(device)
                         for k, v in inputs.items()}
                targets = targets.float().to(device)

                outputs = model(**inputs)
                outputs = torch.clamp(outputs, 0.0, 1.0)

                loss = loss_fn(outputs, targets)

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    losses.append(loss.item())
                    all_predictions.append(outputs)
                    all_targets.append(targets)
                    processed_samples += targets.shape[0]

                del inputs, targets, outputs, loss

            except Exception as e:
                print(f"Error in {dataset_name.lower()} batch {batch_idx}: {e}")
                continue

    if not all_predictions:
        print(f"No valid predictions for {dataset_name} set!")
        return None

    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    avg_loss = np.mean(losses)
    metrics = calculate_accuracy_metrics(all_predictions, all_targets)

    print(f"  Processed {processed_samples} samples")

    return {
        'loss': avg_loss,
        'metrics': metrics,
        'predictions': all_predictions,
        'targets': all_targets,
        'processed_samples': processed_samples
    }

# --------------------------
# Training Loop with LoRA
# --------------------------

class SmoothMSELoss(nn.Module):
    def __init__(self, smoothing=0.01):
        super().__init__()
        self.smoothing = smoothing
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        target_smooth = target * (1 - self.smoothing) + self.smoothing / 2
        return self.mse(pred, target_smooth)

loss_fn = SmoothMSELoss()

lora_params = list(model.blip.parameters())
regressor_params = list(model.regressor.parameters())
all_params = lora_params + regressor_params

optimizer = torch.optim.AdamW(all_params, lr=1e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

num_epochs = 50
best_val_loss = float('inf')
patience_counter = 0
max_patience = 4

print("Starting LoRA fine-tuning...")
print(f"Training {trainable_params:,} parameters out of {total_params:,}")
print("-" * 50)

for epoch in range(num_epochs):
    model.train()
    train_losses = []
    train_samples_processed = 0
    train_batches = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        try:
            inputs = {k: v.to(device).float() if v.dtype == torch.float16 else v.to(device)
                     for k, v in inputs.items()}
            targets = targets.float().to(device)

            optimizer.zero_grad()

            outputs = model(**inputs)
            outputs = torch.clamp(outputs, 0.0, 1.0)

            loss = loss_fn(outputs, targets)

            if torch.isnan(loss) or torch.isinf(loss) or loss.item() > 10:
                print(f"Skipping batch {batch_idx} due to invalid loss: {loss.item()}")
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)

            optimizer.step()

            train_losses.append(loss.item())
            train_samples_processed += targets.shape[0]
            train_batches += 1

            del inputs, targets, outputs, loss

            if batch_idx % 10 == 0:
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        except Exception as e:
            print(f"Error in training batch {batch_idx}: {e}")
            continue

    # Validation phase
    model.eval()
    val_losses = []
    val_samples_processed = 0
    val_batches = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            try:
                inputs = {k: v.to(device).float() if v.dtype == torch.float16 else v.to(device)
                         for k, v in inputs.items()}
                targets = targets.float().to(device)

                outputs = model(**inputs)
                outputs = torch.clamp(outputs, 0.0, 1.0)

                loss = loss_fn(outputs, targets)

                if not (torch.isnan(loss) or torch.isinf(loss)):
                    val_losses.append(loss.item())
                    val_samples_processed += targets.shape[0]
                    val_batches += 1

                del inputs, targets, outputs, loss

            except Exception as e:
                print(f"Error in validation batch {batch_idx}: {e}")
                continue

    if train_batches > 0 and val_batches > 0:
        avg_train_loss = sum(train_losses) / len(train_losses)
        avg_val_loss = sum(val_losses) / len(val_losses)

        train_loss_pct = (avg_train_loss ** 0.5) * 100
        val_loss_pct = (avg_val_loss ** 0.5) * 100

        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss_pct:.2f}% (RMSE), Raw: {avg_train_loss:.6f}")
        print(f"  Val Loss: {val_loss_pct:.2f}% (RMSE), Raw: {avg_val_loss:.6f}")
        print(f"  Train: {train_samples_processed} samples in {train_batches} batches")
        print(f"  Val: {val_samples_processed} samples in {val_batches} batches")
        print(f"  Current LR: {optimizer.param_groups[0]['lr']:.2e}")

        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_path = "/best_fabric_blip2_adapters_only.pth"
            torch.save({
                'lora_adapters': {k: v for k, v in model.blip.state_dict().items() if 'lora_' in k},
                'regressor_state_dict': model.regressor.state_dict(),
                'lora_config': lora_config,
                'model_config': {
                    'hidden_size': model.blip.base_model.language_model.config.hidden_size,
                    'num_classes': 6,
                    'fiber_columns': fiber_columns
                },
                'epoch': epoch,
                'best_val_loss': best_val_loss
            }, best_model_path)
            print(f"  New best adapters saved! Val loss: {val_loss_pct:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break
    else:
        print(f"Epoch {epoch+1}/{num_epochs}: No valid batches processed")

    print("-" * 50)

print("\nTraining completed! Starting final evaluation on test set...")

# =========================
# FINAL TESTING PHASE
# =========================

print("Loading best adapters for final testing...")
best_model_path = "best_fabric_blip2_adapters_only.pth"
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)

    lora_state_dict = checkpoint['lora_adapters']
    current_state_dict = model.blip.state_dict()
    
    for k, v in lora_state_dict.items():
        if k in current_state_dict:
            current_state_dict[k] = v
    
    model.blip.load_state_dict(current_state_dict)
    model.regressor.load_state_dict(checkpoint['regressor_state_dict'])
    print(f"Loaded best adapters from epoch {checkpoint['epoch']+1}")
else:
    print("WARNING: Best adapters not found, using final model for testing")

test_results = evaluate_model(model, test_loader, loss_fn, device, "Test")

if test_results:
    print("\n" + "="*60)
    print("FINAL TEST RESULTS")
    print("="*60)

    test_loss_pct = (test_results['loss'] ** 0.5) * 100
    metrics = test_results['metrics']

    print(f"Test Loss (RMSE): {test_loss_pct:.2f}%")
    print(f"Test Loss (Raw): {test_results['loss']:.6f}")
    print(f"\nOverall Accuracy Metrics:")
    print(f"  Mean Absolute Error: {metrics['MAE']:.2f} percentage points")
    print(f"  Root Mean Square Error: {metrics['RMSE']:.2f} percentage points")
    print(f"  R² Score: {metrics['R²']:.4f}")
    print(f"  Tolerance Accuracy (±10%): {metrics['Tolerance_Accuracy']:.2f}%")

    print(f"\nPer-Fiber Performance:")
    print("-" * 40)
    for fiber, fiber_metrics in metrics['Fiber_Metrics'].items():
        print(f"{fiber.capitalize():12} | MAE: {fiber_metrics['MAE']:5.2f}% | R²: {fiber_metrics['R²']:6.4f}")

    predictions = test_results['predictions'].cpu().numpy() * 100
    targets = test_results['targets'].cpu().numpy() * 100

    print(f"\nAdditional Statistics:")
    print(f"  Number of test samples: {len(predictions)}")
    print(f"  Average prediction sum: {np.mean(np.sum(predictions, axis=1)):.1f}%")
    print(f"  Average target sum: {np.mean(np.sum(targets, axis=1)):.1f}%")
    print(f"  LoRA Efficiency: {trainable_params/total_params:.4f} of parameters trained")

    print(f"\nSample Predictions (first 10 test samples):")
    print("Format: [Cotton, Wool, Polyester, Linen, Silk, Other]")
    print("-" * 70)
    for i in range(min(10, len(predictions))):
        pred_raw = predictions[i]
        targ = targets[i]
       
        pred_sum = np.sum(pred_raw)
        if pred_sum > 0:
            pred_normalized = (pred_raw / pred_sum) * 100
        else:
            pred_normalized = pred_raw
       
        print(f"Sample {i+1}:")
        print(f"  Predicted (Raw):  [{pred_raw[0]:5.1f}, {pred_raw[1]:5.1f}, {pred_raw[2]:5.1f}, {pred_raw[3]:5.1f}, {pred_raw[4]:5.1f}, {pred_raw[5]:5.1f}] Sum: {np.sum(pred_raw):5.1f}%")
        print(f"  Predicted (Norm): [{pred_normalized[0]:5.1f}, {pred_normalized[1]:5.1f}, {pred_normalized[2]:5.1f}, {pred_normalized[3]:5.1f}, {pred_normalized[4]:5.1f}, {pred_normalized[5]:5.1f}] Sum: {np.sum(pred_normalized):5.1f}%")
        print(f"  Actual:           [{targ[0]:5.1f}, {targ[1]:5.1f}, {targ[2]:5.1f}, {targ[3]:5.1f}, {targ[4]:5.1f}, {targ[5]:5.1f}] Sum: {np.sum(targ):5.1f}%")
       
        mae_raw = np.mean(np.abs(pred_raw - targ))
        mae_normalized = np.mean(np.abs(pred_normalized - targ))
        print(f"  Sample MAE (Raw): {mae_raw:.2f}% | MAE (Normalized): {mae_normalized:.2f}%")
        print()

# =========================
# SPACE-EFFICIENT FINAL SAVE
# =========================

save_path = "fabric_blip2_adapters_only.pth"

final_save_dict = {
    'lora_adapters': {k: v for k, v in model.blip.state_dict().items() if 'lora_' in k},
    'regressor_state_dict': model.regressor.state_dict(),
    'lora_config': lora_config,
    'model_config': {
        'hidden_size': model.blip.base_model.language_model.config.hidden_size,
        'num_classes': 6,
        'fiber_columns': fiber_columns,
        'base_model_name': "Salesforce/blip2-opt-2.7b"
    },
    'training_complete': True,
    'test_results': test_results['metrics'] if test_results else None,
    'trainable_params': trainable_params,
    'total_params': total_params
}

torch.save(final_save_dict, save_path)

file_size = os.path.getsize(save_path) / (1024 * 1024)
print(f"\nFinal LoRA adapters and regressor saved to {save_path}")
print(f"File size: {file_size:.2f} MB (only adapters + regressor)")
print(f"Training completed with {trainable_params:,} trainable parameters ({trainable_params/total_params:.4f} of total)")

print("\n" + "="*60)
print("INFERENCE LOADING INSTRUCTIONS")
print("="*60)
print("""
To load the model for inference (minimal memory usage):

from transformers import Blip2Processor, Blip2ForConditionalGeneration
from peft import LoraConfig, get_peft_model
import torch
import torch.nn as nn

checkpoint = torch.load("fabric_blip2_adapters_only.pth", map_location='cpu', weights_only=False)

base_model = Blip2ForConditionalGeneration.from_pretrained(
    checkpoint['model_config']['base_model_name'],
    torch_dtype=torch.float32
)

lora_config = checkpoint['lora_config']
model_with_lora = get_peft_model(base_model, lora_config)

lora_state_dict = checkpoint['lora_adapters']
current_state_dict = model_with_lora.state_dict()

for k, v in lora_state_dict.items():
    if k in current_state_dict:
        current_state_dict[k] = v

model_with_lora.load_state_dict(current_state_dict)

hidden_size = checkpoint['model_config']['hidden_size']
regressor = nn.Sequential(
    nn.Dropout(0.1),
    nn.Linear(hidden_size, hidden_size // 2),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(hidden_size // 2, 6),
    nn.Sigmoid()
)
regressor.load_state_dict(checkpoint['regressor_state_dict'])

class Blip2WithLoRARegressor(nn.Module):
    def __init__(self, blip_model, regressor):
        super().__init__()
        self.blip = blip_model
        self.regressor = regressor
    
    def forward(self, pixel_values, input_ids, attention_mask):
        vision_outputs = self.blip.base_model.vision_model(pixel_values=pixel_values)
        image_embeds = vision_outputs[0]
        image_attention_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=image_embeds.device)
        query_tokens = self.blip.base_model.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_outputs = self.blip.base_model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True,
        )
        query_output = query_outputs.last_hidden_state
        language_model_inputs = self.blip.base_model.language_projection(query_output)
        pooled_output = language_model_inputs[:, 0, :]
        return self.regressor(pooled_output)

inference_model = Blip2WithLoRARegressor(model_with_lora, regressor)
inference_model.eval()

processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
""")

/home/ccbd/Desktop/fabric/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Preparing datasets...
--------------------------------------------------


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Training Set: 1584 valid images found (out of 1584 total)
Validation Set: 198 valid images found (out of 198 total)
Test Set: 201 valid images found (out of 201 total)
--------------------------------------------------
Dataset Summary:
  Train: 1584 samples
  Validation: 198 samples
  Test: 201 samples
  Total: 1983 samples
--------------------------------------------------
DataLoader Info:
  Train batches: 792 (covers 1584 samples)
  Val batches: 99 (covers 198 samples)
  Test batches: 101 (covers 201 samples)
--------------------------------------------------
Using device: cuda


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 10.06it/s]


Applying LoRA to BLIP2 model...
Total parameters: 3,777,779,712
Trainable parameters: 33,017,856
Trainable parameters ratio: 0.0087
Model created with LoRA and regression head in float32 precision
Starting LoRA fine-tuning...
Training 33,017,856 parameters out of 3,777,779,712
--------------------------------------------------
Epoch 1/50:
  Train Loss: 28.12% (RMSE), Raw: 0.079073
  Val Loss: 26.62% (RMSE), Raw: 0.070869
  Train: 1584 samples in 792 batches
  Val: 198 samples in 99 batches
  Current LR: 1.00e-04
  New best adapters saved! Val loss: 26.62%
--------------------------------------------------
Epoch 2/50:
  Train Loss: 24.17% (RMSE), Raw: 0.058416
  Val Loss: 26.02% (RMSE), Raw: 0.067702
  Train: 1584 samples in 792 batches
  Val: 198 samples in 99 batches
  Current LR: 1.00e-04
  New best adapters saved! Val loss: 26.02%
--------------------------------------------------
Epoch 3/50:
  Train Loss: 20.92% (RMSE), Raw: 0.043780
  Val Loss: 26.05% (RMSE), Raw: 0.067849
  Train